# 🎾 Game Analytics: SQL Analysis Queries
## Notebook 3 — All 20 Analysis Queries (PostgreSQL)

This notebook executes all 20 required SQL queries across three categories:

### A. competitions Analysis (7 queries)
### B. Venue Analysis (7 queries)
### C. Competitor Ranking Analysis (6 queries)

> **Database:** PostgreSQL
> All queries use standard PostgreSQL syntax.

## 1. Connect to Database

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Database Configuration (PostgreSQL)
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "your_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "tennis_db")

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

# Test connection
with engine.connect() as conn:
    print("Connected to PostgreSQL database:", DB_NAME)

## A. competitions Analysis

### A1. List all competitions along with their category name

In [ ]:
query = """
SELECT
    c.competition_id,
    c.competition_name,
    cat.category_name,
    c.type,
    c.gender
FROM competitions c
JOIN categories cat ON c.category_id = cat.category_id
ORDER BY cat.category_name, c.competition_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### A2. Count the number of competitions in each category

In [ ]:
query = """
SELECT
    cat.category_name,
    COUNT(c.competition_id) AS competition_count
FROM categories cat
LEFT JOIN competitions c ON cat.category_id = c.category_id
GROUP BY cat.category_id, cat.category_name
ORDER BY competition_count DESC;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### A3. Find all competitions of type 'doubles'

In [ ]:
query = """
SELECT
    competition_id,
    competition_name,
    type,
    gender
FROM competitions
WHERE type = 'doubles'
ORDER BY competition_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### A4. Get competitions that belong to a specific category (e.g., ITF Men)

In [ ]:
query = """
SELECT
    c.competition_id,
    c.competition_name,
    c.type,
    c.gender,
    cat.category_name
FROM competitions c
JOIN categories cat ON c.category_id = cat.category_id
WHERE cat.category_name = 'ITF Men'
ORDER BY c.competition_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### A5. Identify parent competitions and their sub-competitions

In [ ]:
query = """
SELECT
    parent.competition_id   AS parent_id,
    parent.competition_name AS parent_name,
    child.competition_id    AS child_id,
    child.competition_name  AS child_name
FROM competitions child
JOIN competitions parent ON child.parent_id = parent.competition_id
ORDER BY parent.competition_name, child.competition_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### A6. Analyze the distribution of competition types by category

In [ ]:
query = """
SELECT
    cat.category_name,
    c.type,
    COUNT(*) AS count
FROM competitions c
JOIN categories cat ON c.category_id = cat.category_id
GROUP BY cat.category_id, cat.category_name, c.type
ORDER BY cat.category_name, c.type;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### A7. List all top-level competitions (no parent)

In [ ]:
query = """
SELECT
    competition_id,
    competition_name,
    type,
    gender
FROM competitions
WHERE parent_id IS NULL
ORDER BY competition_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

## B. Venue Analysis

### B1. List all venues along with their associated complex name

In [ ]:
query = """
SELECT
    v.venue_id,
    v.venue_name,
    v.city_name,
    v.country_name,
    cx.complex_name
FROM venues v
JOIN complexes cx ON v.complex_id = cx.complex_id
ORDER BY cx.complex_name, v.venue_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### B2. Count the number of venues in each complex

In [ ]:
query = """
SELECT
    cx.complex_name,
    COUNT(v.venue_id) AS venue_count
FROM complexes cx
LEFT JOIN venues v ON cx.complex_id = v.complex_id
GROUP BY cx.complex_id, cx.complex_name
ORDER BY venue_count DESC;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### B3. Get details of venues in a specific country (e.g., Chile)

In [ ]:
query = """
SELECT
    v.venue_id,
    v.venue_name,
    v.city_name,
    v.country_name,
    v.timezone,
    cx.complex_name
FROM venues v
JOIN complexes cx ON v.complex_id = cx.complex_id
WHERE v.country_name = 'Chile'
ORDER BY v.city_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### B4. Identify all venues and their timezones

In [ ]:
query = """
SELECT
    venue_id,
    venue_name,
    city_name,
    country_name,
    timezone
FROM venues
ORDER BY country_name, city_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### B5. Find complexes that have more than one venue

In [ ]:
query = """
SELECT
    cx.complex_name,
    COUNT(v.venue_id) AS venue_count
FROM complexes cx
JOIN venues v ON cx.complex_id = v.complex_id
GROUP BY cx.complex_id, cx.complex_name
HAVING COUNT(v.venue_id) > 1
ORDER BY venue_count DESC;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### B6. List venues grouped by country

In [ ]:
query = """
SELECT
    country_name,
    venue_id,
    venue_name,
    city_name
FROM venues
ORDER BY country_name, city_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### B7. Find all venues for a specific complex (e.g., Nacional)

In [ ]:
query = """
SELECT
    v.venue_id,
    v.venue_name,
    v.city_name,
    v.country_name,
    v.timezone
FROM venues v
JOIN complexes cx ON v.complex_id = cx.complex_id
WHERE cx.complex_name = 'Nacional'
ORDER BY v.venue_name;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

## C. Competitor Ranking Analysis

### C1. Get all competitors with their rank and points

In [ ]:
query = """
SELECT
    comp.name,
    comp.country,
    cr.rank,
    cr.points,
    cr.movement,
    cr.competitions_played
FROM competitor_rankings cr
JOIN competitors comp ON cr.competitor_id = comp.competitor_id
ORDER BY cr.rank;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### C2. Find competitors ranked in the top 5

In [ ]:
query = """
SELECT
    comp.name,
    comp.country,
    comp.country_code,
    cr.rank,
    cr.points
FROM competitor_rankings cr
JOIN competitors comp ON cr.competitor_id = comp.competitor_id
WHERE cr.rank <= 5
ORDER BY cr.rank;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### C3. List competitors with no rank movement (stable rank)

In [ ]:
query = """
SELECT
    comp.name,
    comp.country,
    cr.rank,
    cr.points,
    cr.movement
FROM competitor_rankings cr
JOIN competitors comp ON cr.competitor_id = comp.competitor_id
WHERE cr.movement = 0
ORDER BY cr.rank;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### C4. Get the total points of competitors from a specific country (e.g., Croatia)

In [ ]:
query = """
SELECT
    comp.country,
    SUM(cr.points) AS total_points,
    COUNT(comp.competitor_id) AS competitor_count
FROM competitor_rankings cr
JOIN competitors comp ON cr.competitor_id = comp.competitor_id
WHERE comp.country = 'Croatia'
GROUP BY comp.country;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### C5. Count the number of competitors per country

In [ ]:
query = """
SELECT
    country,
    COUNT(*) AS competitor_count
FROM competitors
GROUP BY country
ORDER BY competitor_count DESC;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

### C6. Find the competitor with the highest points in the current week

In [ ]:
query = """
SELECT
    comp.name,
    comp.country,
    cr.rank,
    cr.points,
    cr.competitions_played
FROM competitor_rankings cr
JOIN competitors comp ON cr.competitor_id = comp.competitor_id
ORDER BY cr.points DESC
LIMIT 1;
"""
df = pd.read_sql(text(query), engine)
print(f"({len(df)} rows)")
df

## All 20 Queries Executed!

### Query Summary

| # | Category | Query |
|---|----------|-------|
| A1-A7 | competitions | List, count, filter, parent-child, type distribution, top-level |
| B1-B7 | venues | List, count, filter by country/complex, timezones, multi-venue |
| C1-C6 | Rankings | All competitors, top 5, stable rank, country points, count, highest |

### Next Steps
-> Open **Notebook 4 - Streamlit App** to launch the interactive dashboard.